In [4]:

import pandas as pd
import requests
from pathlib import Path

# ==========================================
# 1. CONFIGURATION
# ==========================================

YEAR = 2026

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "dim_calendar.csv"
OUTPUT_PARQUET = OUTPUT_DIR / "dim_calendar.parquet"

API_URL = (
    f"https://indonesian-holidays.ahrulsyamil.com/"
    f"holidays/{YEAR}"
)


# ==========================================
# 2. GENERATE COMPLETE CALENDAR
# ==========================================

dates = pd.date_range(
    start=f"{YEAR}-01-01",
    end=f"{YEAR}-12-31",
    freq="D"
)

calendar = pd.DataFrame({
    "attendance_date": dates
})

print("Calendar generated successfully!")
print("Total dates:", len(calendar))


# ==========================================
# 3. GET INDONESIAN HOLIDAYS
# ==========================================

print("\nFetching Indonesian holidays...")

response = requests.get(
    API_URL,
    timeout=30
)

response.raise_for_status()

json_data = response.json()

if "data" not in json_data:
    raise ValueError(
        "Holiday API response does not contain 'data'"
    )

holidays = pd.DataFrame(
    json_data["data"]
)

required_holiday_columns = [
    "date",
    "name",
    "type",
    "is_national"
]

missing_columns = [
    col for col in required_holiday_columns
    if col not in holidays.columns
]

if missing_columns:
    raise ValueError(
        f"Missing API columns: {missing_columns}"
    )

if holidays.empty:
    print("WARNING: No holiday records returned by API.")

else:

    holidays["date"] = pd.to_datetime(
        holidays["date"],
        errors="raise"
    ).dt.normalize()

    holidays["name"] = (
        holidays["name"]
        .fillna("")
        .astype("string")
        .str.strip()
    )

    holidays["type"] = (
        holidays["type"]
        .fillna("")
        .astype("string")
        .str.strip()
    )


# ==========================================
# 4. STANDARDIZE HOLIDAY FLAGS
# ==========================================

def convert_to_boolean(series):

    normalized = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    true_values = [
        "true",
        "1",
        "yes",
        "y"
    ]

    false_values = [
        "false",
        "0",
        "no",
        "n",
        ""
    ]

    result = normalized.map(
        lambda x: (
            True if x in true_values
            else False if x in false_values
            else pd.NA
        )
    )

    return result.astype("boolean")


if not holidays.empty:

    holidays["is_national"] = convert_to_boolean(
        holidays["is_national"]
    )

    if holidays["is_national"].isna().any():
        raise ValueError(
            "Invalid values found in is_national"
        )


# ==========================================
# 5. GROUP HOLIDAYS BY DATE
# ==========================================

# One date may have multiple holiday records.
# Group them into one calendar record.

def combine_unique(values):

    unique_values = (
        values.astype("string")
        .dropna()
        .str.strip()
    )

    unique_values = unique_values[
        unique_values.ne("")
    ]

    return " | ".join(
        unique_values.drop_duplicates().tolist()
    )


if not holidays.empty:

    holidays_grouped = (
        holidays.groupby("date")
        .agg(
            holiday_name=(
                "name",
                combine_unique
            ),

            holiday_type=(
                "type",
                combine_unique
            ),

            is_national_holiday=(
                "is_national",
                "any"
            )
        )
        .reset_index()
    )

    holidays_grouped = (
        holidays_grouped.rename(
            columns={
                "date": "attendance_date"
            }
        )
    )

else:

    holidays_grouped = pd.DataFrame(
        columns=[
            "attendance_date",
            "holiday_name",
            "holiday_type",
            "is_national_holiday"
        ]
    )


# ==========================================
# 6. MERGE CALENDAR + HOLIDAYS
# ==========================================

calendar = calendar.merge(
    holidays_grouped,
    on="attendance_date",
    how="left",
    validate="one_to_one"
)

# Fill dates without holidays

calendar["holiday_name"] = (
    calendar["holiday_name"]
    .fillna("")
    .astype("string")
)

calendar["holiday_type"] = (
    calendar["holiday_type"]
    .fillna("")
    .astype("string")
)

calendar["is_national_holiday"] = (
    calendar["is_national_holiday"]
    .fillna(False)
    .astype("boolean")
)


# ==========================================
# 7. CREATE CALENDAR ATTRIBUTES
# ==========================================

day_mapping = {
    0: "Senin",
    1: "Selasa",
    2: "Rabu",
    3: "Kamis",
    4: "Jumat",
    5: "Sabtu",
    6: "Minggu"
}

month_mapping = {
    1: "Januari",
    2: "Februari",
    3: "Maret",
    4: "April",
    5: "Mei",
    6: "Juni",
    7: "Juli",
    8: "Agustus",
    9: "September",
    10: "Oktober",
    11: "November",
    12: "Desember"
}

calendar["year"] = (
    calendar["attendance_date"].dt.year
)

calendar["quarter"] = (
    calendar["attendance_date"].dt.quarter
)

calendar["month"] = (
    calendar["attendance_date"].dt.month
)

calendar["month_name"] = (
    calendar["month"].map(month_mapping)
)

calendar["day"] = (
    calendar["attendance_date"].dt.day
)

calendar["day_of_week"] = (
    calendar["attendance_date"].dt.dayofweek
)

calendar["day_name"] = (
    calendar["day_of_week"].map(day_mapping)
)


# ==========================================
# 8. CREATE WEEKEND ATTRIBUTES
# ==========================================

calendar["is_weekend"] = (
    calendar["day_of_week"] >= 5
)

calendar["day_type"] = "Weekday"

calendar.loc[
    calendar["is_weekend"],
    "day_type"
] = "Weekend"


# ==========================================
# 9. CREATE JOINT LEAVE ATTRIBUTE
# ==========================================

# Joint leave is kept separate from
# national holidays.

calendar["is_joint_leave"] = (
    calendar["holiday_type"]
    .str.contains(
        "joint_leave",
        case=False,
        na=False,
        regex=False
    )
)


# ==========================================
# 10. CREATE GENERAL HOLIDAY ATTRIBUTE
# ==========================================

calendar["is_any_holiday"] = (
    calendar["is_national_holiday"]
    |
    calendar["is_joint_leave"]
)


# ==========================================
# 11. CREATE WEEK ATTRIBUTES
# ==========================================

# Monday = beginning of the week
# Sunday = end of the week

calendar["week_start"] = (
    calendar["attendance_date"]
    -
    pd.to_timedelta(
        calendar["day_of_week"],
        unit="D"
    )
)

calendar["week_end"] = (
    calendar["week_start"]
    +
    pd.Timedelta(days=6)
)

# ISO week information

iso_calendar = (
    calendar["attendance_date"]
    .dt.isocalendar()
)

calendar["week_year"] = (
    iso_calendar["year"]
)

calendar["week_number"] = (
    iso_calendar["week"]
)

calendar["year_week"] = (
    calendar["week_year"].astype("string")
    + "-W"
    + calendar["week_number"]
    .astype("string")
    .str.zfill(2)
)


# ==========================================
# 12. COUNT WEEKLY NATIONAL HOLIDAYS
# ==========================================

# Count only Monday-Friday national holidays.
# Weekend holidays do not reduce FIXED_5 quota.

weekday_national_holidays = (
    calendar[
        calendar["is_national_holiday"]
        &
        calendar["day_of_week"].between(0, 4)
    ]
    .groupby("week_start")
    .size()
    .rename("weekday_national_holidays")
    .reset_index()
)

calendar = calendar.merge(
    weekday_national_holidays,
    on="week_start",
    how="left",
    validate="many_to_one"
)

calendar["weekday_national_holidays"] = (
    calendar["weekday_national_holidays"]
    .fillna(0)
    .astype("int64")
)


# ==========================================
# 13. COUNT WEEKLY JOINT LEAVES
# ==========================================

# Kept separate because the business rule
# for joint leave has not been confirmed.

weekday_joint_leaves = (
    calendar[
        calendar["is_joint_leave"]
        &
        calendar["day_of_week"].between(0, 4)
    ]
    .groupby("week_start")
    .size()
    .rename("weekday_joint_leaves")
    .reset_index()
)

calendar = calendar.merge(
    weekday_joint_leaves,
    on="week_start",
    how="left",
    validate="many_to_one"
)

calendar["weekday_joint_leaves"] = (
    calendar["weekday_joint_leaves"]
    .fillna(0)
    .astype("int64")
)


# ==========================================
# 14. CREATE WEEKLY HOLIDAY SUMMARY
# ==========================================

# This column is useful for FIXED_5 calculation.
# It counts national holidays on weekdays.

calendar["weekly_national_holiday_count"] = (
    calendar["weekday_national_holidays"]
)


# ==========================================
# 15. DATA VALIDATION
# ==========================================

print("\n==========================================")
print("CALENDAR DATA VALIDATION")
print("==========================================")


# Check total dates

expected_dates = (
    366 if pd.Timestamp(f"{YEAR}-12-31").is_leap_year
    else 365
)

if len(calendar) != expected_dates:
    raise ValueError(
        f"Unexpected date count: {len(calendar)}"
    )


# Check duplicate dates

duplicate_dates = (
    calendar["attendance_date"].duplicated().sum()
)

if duplicate_dates > 0:
    raise ValueError(
        f"Duplicate dates found: {duplicate_dates}"
    )


# Check missing dates

missing_dates = (
    calendar["attendance_date"].isna().sum()
)

if missing_dates > 0:
    raise ValueError(
        f"Missing dates found: {missing_dates}"
    )


# Check date range

if (
    calendar["attendance_date"].min()
    != pd.Timestamp(f"{YEAR}-01-01")
):
    raise ValueError("Incorrect calendar start date")

if (
    calendar["attendance_date"].max()
    != pd.Timestamp(f"{YEAR}-12-31")
):
    raise ValueError("Incorrect calendar end date")


print("Expected dates:", expected_dates)
print("Actual dates:", len(calendar))
print("Duplicate dates:", duplicate_dates)
print("Missing dates:", missing_dates)
print("Date range: VALID")


# ==========================================
# 16. REORDER COLUMNS
# ==========================================

column_order = [
    "attendance_date",

    "year",
    "quarter",
    "month",
    "month_name",
    "day",

    "day_of_week",
    "day_name",
    "day_type",
    "is_weekend",

    "holiday_name",
    "holiday_type",
    "is_national_holiday",
    "is_joint_leave",
    "is_any_holiday",

    "week_start",
    "week_end",
    "week_year",
    "week_number",
    "year_week",

    "weekday_national_holidays",
    "weekday_joint_leaves",
    "weekly_national_holiday_count"
]

calendar = calendar[column_order]


# ==========================================
# 17. EXPORT CALENDAR
# ==========================================

calendar.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

calendar.to_parquet(
    OUTPUT_PARQUET,
    index=False
)


# ==========================================
# 18. SUMMARY REPORT
# ==========================================

print("\n==========================================")
print("PROCESS 9 COMPLETED")
print("CALENDAR DIMENSION")
print("==========================================")


print("\nTOTAL DATES:")
print(len(calendar))


print("\nNATIONAL HOLIDAYS:")
print(
    calendar["is_national_holiday"].sum()
)


print("\nJOINT LEAVE:")
print(
    calendar["is_joint_leave"].sum()
)


print("\nWEEKENDS:")
print(
    calendar["is_weekend"].sum()
)


print("\nWEEKDAY NATIONAL HOLIDAYS:")
print(
    calendar.loc[
        calendar["is_national_holiday"]
        &
        calendar["day_of_week"].between(0, 4),
        "attendance_date"
    ].nunique()
)


print("\nWEEKDAY JOINT LEAVES:")
print(
    calendar.loc[
        calendar["is_joint_leave"]
        &
        calendar["day_of_week"].between(0, 4),
        "attendance_date"
    ].nunique()
)


print("\nWEEKLY HOLIDAY SUMMARY:")

weekly_summary = (
    calendar[
        [
            "week_start",
            "weekday_national_holidays",
            "weekday_joint_leaves"
        ]
    ]
    .drop_duplicates(subset=["week_start"])
    .sort_values("week_start")
)

print(
    weekly_summary.to_string(index=False)
)


print("\nOUTPUT FILES:")
print("CSV:", OUTPUT_CSV)
print("PARQUET:", OUTPUT_PARQUET)


print("\n==========================================")
print("PROCESS 9 SUCCESSFULLY FINISHED")
print("==========================================")


Calendar generated successfully!
Total dates: 365

Fetching Indonesian holidays...

CALENDAR DATA VALIDATION
Expected dates: 365
Actual dates: 365
Duplicate dates: 0
Missing dates: 0
Date range: VALID

PROCESS 9 COMPLETED
CALENDAR DIMENSION

TOTAL DATES:
365

NATIONAL HOLIDAYS:
17

JOINT LEAVE:
8

WEEKENDS:
104

WEEKDAY NATIONAL HOLIDAYS:
13

WEEKDAY JOINT LEAVES:
8

WEEKLY HOLIDAY SUMMARY:
week_start  weekday_national_holidays  weekday_joint_leaves
2025-12-29                          1                     0
2026-01-05                          0                     0
2026-01-12                          1                     0
2026-01-19                          0                     0
2026-01-26                          0                     0
2026-02-02                          0                     0
2026-02-09                          0                     0
2026-02-16                          1                     1
2026-02-23                          0                     0
2026-0

C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_1324\2246498835.py:306: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar["attendance_date"].dt.dayofweek
